In [0]:
import os, sys, datetime as dt

# Make the repo's src folder importable (notebook sits in notebooks/02_ingestion)
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(os.path.join(repo_root, "src"))
import importlib, entsoe_client
importlib.reload(entsoe_client)          # pick up edits to src/entsoe_client.py
from entsoe_client import EntsoeClient

# Token: real one once stored as a secret, otherwise a dummy for the dry run
try:
    token = dbutils.secrets.get("power-prices", "entsoe-token")
    mode = "LIVE"
except Exception:
    token = "dummy-token-for-dry-run"
    mode = "DRY RUN (no token yet)"
print("Mode:", mode)

client = EntsoeClient(token=token, spark=spark)

start = dt.datetime(2025, 6, 1, 22, 0, tzinfo=dt.timezone.utc)  # 2025-06-02 00:00 Berlin time
end   = start + dt.timedelta(days=1)

outcome, xml, path = client.fetch("day_ahead_price", "DE_LU", "10Y1001A1001A82H", start, end)
print("Outcome:", outcome, "| file:", path)
if xml:
    print(xml[:400].decode("utf-8", errors="replace"))

print("Log rows written:", client.flush_log())
display(spark.sql("""
    SELECT called_at, dataset, series_key, http_status, outcome, bytes, duration_ms, file_path, error_message
    FROM power_prices.ops.api_call_log
    ORDER BY called_at DESC
    LIMIT 5
"""))